In [ ]:
# ==============================
# PHASE 04 - STEP 01
# Dataset Split Verification (Video-Level Safety Check)
# ==============================

import os
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from google.colab import drive

# 1. SETUP PATHS
drive.mount('/content/drive')
BASE_PATH = "/content/drive/MyDrive/Human-Fall-Recognition"
SEQ_PATH = os.path.join(BASE_PATH, "Sequences")
SPLIT_PATH = os.path.join(BASE_PATH, "Data/Split_Data")
TEST_PATH = os.path.join(BASE_PATH, "Data/Test_Set_Locked")
RESULT_PATH = os.path.join(BASE_PATH, "Results", "Metrics")

os.makedirs(SPLIT_PATH, exist_ok=True)
os.makedirs(TEST_PATH, exist_ok=True)
os.makedirs(RESULT_PATH, exist_ok=True)

# 2. LOAD DATA
X = np.load(os.path.join(SEQ_PATH, "X.npy"))
y = np.load(os.path.join(SEQ_PATH, "y.npy"))
video_ids = np.load(os.path.join(SEQ_PATH, "video_ids.npy"))

# 3. VIDEO-LEVEL SPLIT
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_val_idx, test_idx = next(gss_test.split(X, y, groups=video_ids))

gss_val = GroupShuffleSplit(n_splits=1, test_size=0.1765, random_state=42)
train_idx, val_idx = next(
    gss_val.split(
        X[train_val_idx],
        y[train_val_idx],
        groups=video_ids[train_val_idx]
    )
)

train_idx = train_val_idx[train_idx]
val_idx = train_val_idx[val_idx]

# 4. SAFETY CHECK
train_videos = set(video_ids[train_idx])
val_videos = set(video_ids[val_idx])
test_videos = set(video_ids[test_idx])

assert train_videos.isdisjoint(val_videos)
assert train_videos.isdisjoint(test_videos)
assert val_videos.isdisjoint(test_videos)

# 5. SAVE SPLITS
np.save(os.path.join(SPLIT_PATH, "X_train.npy"), X[train_idx])
np.save(os.path.join(SPLIT_PATH, "y_train.npy"), y[train_idx])
np.save(os.path.join(SPLIT_PATH, "X_val.npy"), X[val_idx])
np.save(os.path.join(SPLIT_PATH, "y_val.npy"), y[val_idx])

np.save(os.path.join(TEST_PATH, "X_test_locked.npy"), X[test_idx])
np.save(os.path.join(TEST_PATH, "y_test_locked.npy"), y[test_idx])
np.save(os.path.join(TEST_PATH, "groups_test_locked.npy"), video_ids[test_idx])

# 6. THESIS TABLE
split_table = pd.DataFrame({
    "Split": ["Training", "Validation", "Test (Locked)"],
    "Sequences": [len(train_idx), len(val_idx), len(test_idx)],
    "Unique Videos": [len(train_videos), len(val_videos), len(test_videos)],
    "Ratio": ["~70%", "~15%", "~15%"]
})

csv_path = os.path.join(RESULT_PATH, "phase04_dataset_split_summary.csv")
split_table.to_csv(csv_path, index=False)
display(split_table)

In [ ]:
# ==============================
# PHASE 04 - STEP 02
# Baseline LSTM Model Training
# ==============================

import os
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import CSVLogger

# 1. PATHS (Drive already mounted in Step 01)
BASE_PATH = "/content/drive/MyDrive/Human-Fall-Recognition"

DATA_PATH = os.path.join(BASE_PATH, "Data/Split_Data")
MODEL_PATH = os.path.join(BASE_PATH, "Models")
FIG_PATH = os.path.join(BASE_PATH, "Results", "Figures")
METRIC_PATH = os.path.join(BASE_PATH, "Results", "Metrics")

os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(FIG_PATH, exist_ok=True)
os.makedirs(METRIC_PATH, exist_ok=True)

# 2. LOAD DATA
print("Loading data...")
X_train = np.load(os.path.join(DATA_PATH, "X_train.npy"))
y_train = np.load(os.path.join(DATA_PATH, "y_train.npy"))
X_val   = np.load(os.path.join(DATA_PATH, "X_val.npy"))
y_val   = np.load(os.path.join(DATA_PATH, "y_val.npy"))

print("Train:", X_train.shape, y_train.shape)
print("Val  :", X_val.shape, y_val.shape)

# 3. BASELINE MODEL (STANDARD LSTM)
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=X_train.shape[1:]),
    Dropout(0.3),
    LSTM(32),
    Dropout(0.3),
    Dense(4, activation="softmax")
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]  # internal only
)

model.summary()

# 4. TRAIN
csv_logger = CSVLogger(
    os.path.join(METRIC_PATH, "phase04_baseline_history.csv"),
    append=False
)

print("\nStarting training...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    callbacks=[csv_logger],
    verbose=1
)

# 5. SAVE MODEL
model.save(os.path.join(MODEL_PATH, "model_baseline.keras"))

# 6. LOSS CURVE (SHOW + SAVE)
plt.figure()
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Baseline LSTM Training Loss")
plt.legend()
plt.grid(True)
plt.savefig(os.path.join(FIG_PATH, "phase04_baseline_loss.png"), dpi=300)
plt.show()

print("\n✅ PHASE 04 – STEP 02 COMPLETE")
print("Model  → Models/model_baseline.keras")
print("Figure → Results/Figures/phase04_baseline_loss.png")
print("Log    → Results/Metrics/phase04_baseline_history.csv")

In [ ]:
# ==============================
# PHASE 04 - STEP 03
# Enhanced Bi-LSTM Model Training
# ==============================

import os
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, BatchNormalization, GaussianNoise
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import CSVLogger, ReduceLROnPlateau

# 1. PATHS (Drive already mounted)
BASE_PATH = "/content/drive/MyDrive/Human-Fall-Recognition"

DATA_PATH = os.path.join(BASE_PATH, "Data/Split_Data")
MODEL_PATH = os.path.join(BASE_PATH, "Models")
FIG_PATH = os.path.join(BASE_PATH, "Results", "Figures")
METRIC_PATH = os.path.join(BASE_PATH, "Results", "Metrics")

os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(FIG_PATH, exist_ok=True)
os.makedirs(METRIC_PATH, exist_ok=True)

# 2. LOAD DATA
print("Loading data...")
X_train = np.load(os.path.join(DATA_PATH, "X_train.npy"))
y_train = np.load(os.path.join(DATA_PATH, "y_train.npy"))
X_val   = np.load(os.path.join(DATA_PATH, "X_val.npy"))
y_val   = np.load(os.path.join(DATA_PATH, "y_val.npy"))

print("Train:", X_train.shape, y_train.shape)
print("Val  :", X_val.shape, y_val.shape)

# 3. ENHANCED MODEL (Bi-LSTM + Enhancements)
model = Sequential([
    # Input Layer with Noise (Simulates real-world sensor jitter)
    GaussianNoise(0.05, input_shape=X_train.shape[1:]),

    # Bidirectional LSTM Layer 1
    Bidirectional(LSTM(64, return_sequences=True)),
    BatchNormalization(),
    Dropout(0.3),

    # Bidirectional LSTM Layer 2
    Bidirectional(LSTM(32)),
    BatchNormalization(),
    Dropout(0.3),

    # Output Layer
    Dense(4, activation="softmax")
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# 4. CALLBACKS
csv_logger = CSVLogger(
    os.path.join(METRIC_PATH, "phase04_enhanced_history.csv"),
    append=False
)

# Learning Rate Scheduler
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-5,
    verbose=1
)

# 5. TRAIN (30 EPOCHS - Same as Baseline for fair comparison)
print("\nStarting enhanced training...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    callbacks=[csv_logger, lr_scheduler],
    verbose=1
)

# 6. SAVE MODEL
model.save(os.path.join(MODEL_PATH, "model_enhanced.keras"))

# 7. LOSS CURVE
plt.figure()
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Enhanced Bi-LSTM Training Loss")
plt.legend()
plt.grid(True)
plt.savefig(os.path.join(FIG_PATH, "phase04_enhanced_loss.png"), dpi=300)
plt.show()

print("\n✅ PHASE 04 – STEP 03 COMPLETE")
print("Model  → Models/model_enhanced.keras")
print("Figure → Results/Figures/phase04_enhanced_loss.png")
print("Log    → Results/Metrics/phase04_enhanced_history.csv")

In [ ]:
# ==============================
# PHASE 04 - STEP 04
# Final Loss Comparison
# ==============================

import os
import pandas as pd
import matplotlib.pyplot as plt

# 1. PATHS
BASE_PATH = "/content/drive/MyDrive/Human-Fall-Recognition"
METRIC_PATH = os.path.join(BASE_PATH, "Results", "Metrics")
FIG_PATH = os.path.join(BASE_PATH, "Results", "Figures")

os.makedirs(FIG_PATH, exist_ok=True)

# 2. LOAD DATA
baseline_df = pd.read_csv(os.path.join(METRIC_PATH, "phase04_baseline_history.csv"))
enhanced_df = pd.read_csv(os.path.join(METRIC_PATH, "phase04_enhanced_history.csv"))

# 3. PLOT SIDE-BY-SIDE (TRAIN vs VAL)
# This avoids the "Accuracy Trap" but keeps the layout clean.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- PLOT 1: TRAINING LOSS (Learning Speed) ---
ax1.plot(baseline_df['loss'], label='Baseline (Standard)', linestyle='--', color='red', alpha=0.7)
ax1.plot(enhanced_df['loss'], label='Enhanced (Bi-LSTM+Noise)', color='green', linewidth=2)
ax1.set_title('Training Loss (Learning Speed)')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

# --- PLOT 2: VALIDATION LOSS (Generalization - The Key Metric) ---
ax2.plot(baseline_df['val_loss'], label='Baseline (Standard)', linestyle='--', color='red', alpha=0.7)
ax2.plot(enhanced_df['val_loss'], label='Enhanced (Bi-LSTM+Noise)', color='green', linewidth=2)
ax2.set_title('Validation Loss (Generalization)')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

# 4. SAVE
plt.tight_layout()
save_path = os.path.join(FIG_PATH, "phase04_final_loss_comparison.png")
plt.savefig(save_path, dpi=300)
plt.show()

print("\n✅ PHASE 04 COMPLETE")
print(f"Final Graph Saved → {save_path}")
print("Ready for PHASE 05 (Accuracy, Confusion Matrix, and Final Test)")